# Grazing-Incidence sin2psi Strain Analysis

Build a GI polar map, fit a reflection across chi sectors, then regress
d-spacing against sin2(psi). Stress is only meaningful after choosing
material-specific elastic constants and confirming the geometry.


In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from xrd_tools.analysis import Sin2PsiPlan, run_sin2psi
from xrd_tools.core.containers import IntegrationResult2D
from xrd_tools.gui.widgets import PeakFitControls


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
q_range = widgets.FloatRangeSlider(value=(2.75, 3.25), min=2.0, max=4.0, step=0.01, description="q fit", continuous_update=False)
fit_button = widgets.Button(description="Fit sectors", button_style="primary")
display(widgets.VBox([q_range, fit_button]))


In [ ]:
q = np.linspace(2.5, 3.5, 260)
chi = np.linspace(-35.0, 35.0, 25)
intensity = np.empty((q.size, chi.size))
for index, angle in enumerate(chi):
    center = 3.0 + 0.012 * np.sin(np.deg2rad(abs(angle))) ** 2
    intensity[:, index] = 20 + 300 * np.exp(-0.5 * ((q - center) / 0.025) ** 2)
polar = IntegrationResult2D(q, chi, intensity, unit="q_A^-1", azimuthal_unit="chi_deg")
result = run_sin2psi(Sin2PsiPlan(q_range=tuple(q_range.value), chi_width=7.0), polar).payload
assert np.isfinite(result.d0)
{"d0_A": result.d0, "slope_A": result.slope, "r_squared": result.r_squared}
